# 🌳 Python Binary Tree — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A binary tree is like a family tree — one root at the top,
> each person has at most two children (left and right).
> DFS goes deep into one branch before backtracking.
> BFS visits everyone on the same floor before going down a level.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [What Is a Binary Tree? The Visual Model](#1) |
| 2 | [TreeNode Setup and Helpers](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Max Depth — LC 104](#5) |
| 6 | [Pattern 2: Level Order BFS — LC 102](#6) |
| 7 | [Pattern 3: Invert Tree — LC 226](#7) |
| 8 | [Pattern 4: Balanced Tree — LC 110](#8) |
| 9 | [Pattern 5: Diameter — LC 543](#9) |
| 10 | [The Binary Tree Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>

## 1. What Is a Binary Tree? The Visual Model

```
STRUCTURE

           1          ← root (depth 0)
         /   \
        2     3       ← depth 1
       / \     \
      4   5     6    ← depth 2 (leaves)

  Each node: value, left child, right child
  Leaf: no children (left=None, right=None)
  Height: max depth of any leaf = 2
  Diameter: longest path between any two nodes
            = 4→2→1→3→6 = length 4

DFS TRAVERSAL ORDER

  Pre-order:  root → left → right   [1,2,4,5,3,6]
  In-order:   left → root → right   [4,2,5,1,3,6]  ← sorted for BST
  Post-order: left → right → root   [4,5,2,6,3,1]

BFS (LEVEL ORDER)

  Use a queue. Process level by level.
  Level 0: [1]
  Level 1: [2, 3]
  Level 2: [4, 5, 6]

RECURSION TEMPLATE

  def dfs(node):
      if not node: return base_value     ← base case
      left  = dfs(node.left)             ← recurse left
      right = dfs(node.right)            ← recurse right
      return combine(node.val, left, right)  ← combine
```

<a id='2'></a>

## 2. TreeNode Setup and Helpers

In [ ]:
from collections import deque
from typing import Optional, List

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def make_tree(vals: List) -> Optional[TreeNode]:
    """Build tree from BFS-level list. None = missing node."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

def tree_to_list(root: Optional[TreeNode]) -> List:
    """BFS-level serialization back to list."""
    if not root:
        return []
    result, q = [], deque([root])
    while q:
        node = q.popleft()
        if node:
            result.append(node.val)
            q.append(node.left)
            q.append(node.right)
        else:
            result.append(None)
    while result and result[-1] is None:   # trim trailing Nones
        result.pop()
    return result

# Smoke test
t = make_tree([1,2,3,4,5,None,6])
print(f"round-trip: {tree_to_list(t)}")
print("TreeNode helpers ready.")

<a id='3'></a>

## 3. The Core API — All Operations

```
OPERATION                           COMPLEXITY   WHAT IT DOES
──────────────────────────────────────────────────────────────────
DFS recursive (pre/in/post)         O(n)         visit every node once
DFS iterative (stack)               O(n)         same, explicit stack
BFS level-order (deque)             O(n)         process level by level
Height / max depth                  O(n)         max(left, right) + 1
Check balance                       O(n)         height + -1 sentinel
Diameter at node                    O(n)         left_depth + right_depth
Lowest Common Ancestor              O(n)         left and right both found

THINGS YOU DO NOT DO:
❌  Forget base case: if not node: return 0 (or None)
❌  Return height before checking both subtrees (partial traversal)
❌  Use BFS when DFS with post-order is cleaner (height, diameter)
❌  Access node.val before checking node is not None
```

In [ ]:
from collections import deque

t = make_tree([1,2,3,4,5,None,6])

# Pre-order DFS
def preorder(node):
    if not node: return []
    return [node.val] + preorder(node.left) + preorder(node.right)

# In-order DFS
def inorder(node):
    if not node: return []
    return inorder(node.left) + [node.val] + inorder(node.right)

# Post-order DFS
def postorder(node):
    if not node: return []
    return postorder(node.left) + postorder(node.right) + [node.val]

# BFS level-order
def level_order(node):
    if not node: return []
    result, q = [], deque([node])
    while q:
        level = []
        for _ in range(len(q)):
            n = q.popleft()
            level.append(n.val)
            if n.left:  q.append(n.left)
            if n.right: q.append(n.right)
        result.append(level)
    return result

print(f"pre-order:   {preorder(t)}")
print(f"in-order:    {inorder(t)}")
print(f"post-order:  {postorder(t)}")
print(f"level-order: {level_order(t)}")
print("Core API demo done.")

<a id='4'></a>

## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                    APPROACH
──────────────────────────────────────────────────────────────
height / depth / max path               post-order DFS
level-by-level result / right side view BFS with queue
check property on whole tree            post-order + sentinel
validate BST                            in-order / range DFS
path sum root-to-leaf                   pre-order DFS, carry sum
lowest common ancestor                  post-order, return found nodes
serialize / deserialize                 BFS or pre-order + marker
```

<a id='5'></a>

## 5. 🧩 Pattern 1: Max Depth — LC 104

---

```
PROBLEM:
  Return the maximum depth (number of nodes along the longest root-to-leaf path).

TRICK:
  Post-order DFS. Depth of a node = max(left_depth, right_depth) + 1.
  Base case: None node has depth 0.

SLOW MOTION TRACE on [3,9,20,None,None,15,7]:
         3
        / \
       9  20
          / \
         15   7

  dfs(9)  → max(dfs(None),dfs(None))+1 = max(0,0)+1 = 1
  dfs(15) → 1
  dfs(7)  → 1
  dfs(20) → max(1,1)+1 = 2
  dfs(3)  → max(1,2)+1 = 3
  answer = 3

KEY INSIGHT:
  The +1 accounts for the current node itself.
  Post-order: left and right depths must be known before combining.

TIME:  O(n) — visit every node once
SPACE: O(h) — recursion stack depth, h = tree height
```

In [ ]:
def max_depth(root: Optional[TreeNode]) -> int:
    """
    LC 104 — Maximum Depth of Binary Tree
    Approach: post-order DFS, max(left, right) + 1.
    Time:  O(n) — every node visited once
    Space: O(h) — recursion stack, h = height
    """
    if not root:
        return 0                         # base case: empty subtree has depth 0
    left_d  = max_depth(root.left)       # depth of left subtree
    right_d = max_depth(root.right)      # depth of right subtree
    return max(left_d, right_d) + 1      # +1 for current node

# Slow motion on [3,9,20,None,None,15,7]:
# dfs(9)=1, dfs(20)=max(1,1)+1=2, dfs(3)=max(1,2)+1=3

def test_harness(fn):
    tests = [
        ([3,9,20,None,None,15,7], 3),
        ([1,None,2],             2),
        ([],                     0),
        ([1],                    1),
        ([1,2,3,4,5],            3),
    ]
    passed = 0
    for *inputs, expected in tests:
        root = make_tree(inputs[0])
        got = fn(root)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | tree={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(max_depth)
print("max_depth defined.")

<a id='6'></a>

## 6. 🧩 Pattern 2: Level Order BFS — LC 102

---

```
PROBLEM:
  Return each level's values as a list of lists.

TRICK:
  BFS with a queue. At each step, snapshot queue length = current level size.
  Process exactly that many nodes — their children form the next level.

SLOW MOTION TRACE on [3,9,20,None,None,15,7]:
  q=[3], result=[]
  level 0: size=1, pop 3, push 9,20 → result=[[3]]
  level 1: size=2, pop 9(no children), pop 20(push 15,7) → result=[[3],[9,20]]
  level 2: size=2, pop 15, pop 7 → result=[[3],[9,20],[15,7]]
  done

KEY INSIGHT:
  Snapshotting queue length before the inner loop isolates one level.
  Children appended during the loop belong to the NEXT level.

TIME:  O(n) — every node enqueued and dequeued once
SPACE: O(w) — queue holds at most one full level, w = max width
```

In [ ]:
def level_order_bfs(root: Optional[TreeNode]) -> List[List[int]]:
    """
    LC 102 — Binary Tree Level Order Traversal
    Approach: BFS, snapshot queue length for each level.
    Time:  O(n) — each node processed once
    Space: O(w) — queue at most holds max-width level
    """
    if not root:
        return []
    result = []
    q = deque([root])

    while q:
        level_size = len(q)     # snapshot: how many nodes are on this level
        level = []
        for _ in range(level_size):
            node = q.popleft()
            level.append(node.val)
            if node.left:  q.append(node.left)   # next level
            if node.right: q.append(node.right)  # next level
        result.append(level)

    return result

# Slow motion on [3,9,20,None,None,15,7]:
# level 0: [3], level 1: [9,20], level 2: [15,7]

def test_harness(fn):
    tests = [
        ([3,9,20,None,None,15,7], [[3],[9,20],[15,7]]),
        ([1],                     [[1]]),
        ([],                      []),
        ([1,2,3,4,5],             [[1],[2,3],[4,5]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        root = make_tree(inputs[0])
        got = fn(root)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | tree={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(level_order_bfs)
print("level_order_bfs defined.")

<a id='7'></a>

## 7. 🧩 Pattern 3: Invert Binary Tree — LC 226

---

```
PROBLEM:
  Invert (mirror) a binary tree. Swap left and right children at every node.

TRICK:
  Pre-order or post-order DFS. Swap children, then recurse.
  One line per node: node.left, node.right = node.right, node.left.

SLOW MOTION TRACE on [4,2,7,1,3,6,9]:
  original:       inverted:
       4               4
      / \             / \
     2   7           7   2
    / \ / \         / \ / \
   1  3 6  9       9  6 3  1

  At node 4: swap(2,7) → children=[7,2]
  At node 7: swap(6,9) → children=[9,6]
  At node 2: swap(1,3) → children=[3,1]
  Leaves: swap(None,None) → no-op

KEY INSIGHT:
  Mirroring is symmetric — left and right swap recursively.
  Works top-down (pre-order) or bottom-up (post-order).

TIME:  O(n) — visit every node once
SPACE: O(h) — recursion stack
```

In [ ]:
def invert_tree(root: Optional[TreeNode]) -> Optional[TreeNode]:
    """
    LC 226 — Invert Binary Tree
    Approach: DFS, swap left and right children at every node.
    Time:  O(n) — visit every node once
    Space: O(h) — recursion stack depth
    """
    if not root:
        return None                            # base case
    root.left, root.right = root.right, root.left  # swap children
    invert_tree(root.left)                     # recurse on new left (was right)
    invert_tree(root.right)                    # recurse on new right (was left)
    return root

# Slow motion on [4,2,7,1,3,6,9]:
# node 4: swap(2,7) → [7,2], node 7: swap(6,9) → [9,6], node 2: swap(1,3) → [3,1]
# result: [4,7,2,9,6,3,1]

def test_harness(fn):
    tests = [
        ([4,2,7,1,3,6,9], [4,7,2,9,6,3,1]),
        ([2,1,3],         [2,3,1]),
        ([],              []),
        ([1],             [1]),
    ]
    passed = 0
    for *inputs, expected in tests:
        root = make_tree(inputs[0])
        fn(root)
        got = tree_to_list(root)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | tree={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(invert_tree)
print("invert_tree defined.")

<a id='8'></a>

## 8. 🧩 Pattern 4: Balanced Binary Tree — LC 110

---

```
PROBLEM:
  Determine if a tree is height-balanced (left and right subtrees differ
  in height by at most 1 at every node).

TRICK:
  Post-order DFS returns height OR -1 as an "unbalanced" sentinel.
  Once any subtree returns -1, propagate -1 upward — short-circuit.

SLOW MOTION TRACE on [1,2,2,3,3,None,None,4,4]:
  Left subtree of root is deeper:
  dfs(left of 3) = 2
  dfs(right of 3) = 0
  diff = 2 > 1 → return -1 (unbalanced)
  dfs(2) receives -1 from left → return -1
  dfs(root) receives -1 → return False

KEY INSIGHT:
  Using -1 as sentinel avoids a second pass.
  Combine height computation with balance check in one DFS.

TIME:  O(n) — post-order, each node visited once
SPACE: O(h) — recursion stack
```

In [ ]:
def is_balanced(root: Optional[TreeNode]) -> bool:
    """
    LC 110 — Balanced Binary Tree
    Approach: post-order DFS returns height or -1 sentinel for unbalanced.
    Time:  O(n) — each node visited once
    Space: O(h) — recursion stack
    """
    def dfs(node) -> int:
        if not node:
            return 0                          # empty subtree has height 0
        left_h = dfs(node.left)
        if left_h == -1:
            return -1                         # left already unbalanced — short-circuit
        right_h = dfs(node.right)
        if right_h == -1:
            return -1                         # right already unbalanced
        if abs(left_h - right_h) > 1:
            return -1                         # this node is unbalanced
        return max(left_h, right_h) + 1       # return height for parent

    return dfs(root) != -1

# Slow motion on unbalanced [1,2,2,3,3,None,None,4,4]:
# deep left branch returns -1, propagates up → is_balanced=False

def test_harness(fn):
    tests = [
        ([3,9,20,None,None,15,7],        True),
        ([1,2,2,3,3,None,None,4,4],      False),
        ([],                             True),
        ([1],                            True),
        ([1,2,None,3],                   False),
    ]
    passed = 0
    for *inputs, expected in tests:
        root = make_tree(inputs[0])
        got = fn(root)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | tree={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(is_balanced)
print("is_balanced defined.")

<a id='9'></a>

## 9. 🧩 Pattern 5: Diameter of Binary Tree — LC 543

---

```
PROBLEM:
  Return the length of the longest path between any two nodes.
  The path does not need to pass through the root.

TRICK:
  At each node, diameter through that node = left_depth + right_depth.
  Track global max. Return height (not diameter) to parent.
  Two values flow: height (returned up) and diameter (stored in max).

SLOW MOTION TRACE on [1,2,3,4,5]:
       1
      / \
     2   3
    / \
   4   5

  dfs(4): height=1, diameter_here=0
  dfs(5): height=1, diameter_here=0
  dfs(2): left=1, right=1, diameter_here=2 → max=2, return height=2
  dfs(3): height=1, diameter_here=0
  dfs(1): left=2, right=1, diameter_here=3 → max=3, return height=3
  answer=3  (path: 4→2→1→3 or 5→2→1→3)

KEY INSIGHT:
  Diameter at a node = left_height + right_height (number of edges).
  Height returned upward = max(left, right) + 1 (number of nodes).
  These are two different things — the global max captures the diameter.

TIME:  O(n) — post-order, each node once
SPACE: O(h) — recursion stack
```

In [ ]:
def diameter_of_binary_tree(root: Optional[TreeNode]) -> int:
    """
    LC 543 — Diameter of Binary Tree
    Approach: post-order DFS; track max(left_h + right_h) globally.
    Time:  O(n) — each node visited once
    Space: O(h) — recursion stack
    """
    max_diameter = [0]   # list for mutation inside closure

    def dfs(node) -> int:
        if not node:
            return 0
        left_h  = dfs(node.left)
        right_h = dfs(node.right)
        max_diameter[0] = max(max_diameter[0], left_h + right_h)  # diameter at node
        return max(left_h, right_h) + 1    # height returned to parent

    dfs(root)
    return max_diameter[0]

# Slow motion on [1,2,3,4,5]:
# dfs(4)=1, dfs(5)=1, dfs(2): max=max(0,1+1)=2, return 2
# dfs(3)=1, dfs(1): max=max(2,2+1)=3, return 3
# answer=3

def test_harness(fn):
    tests = [
        ([1,2,3,4,5], 3),
        ([1,2],       1),
        ([1],         0),
        ([],          0),
        ([1,2,None,3,None,4], 3),
    ]
    passed = 0
    for *inputs, expected in tests:
        root = make_tree(inputs[0])
        got = fn(root)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | tree={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(diameter_of_binary_tree)
print("diameter_of_binary_tree defined.")

<a id='10'></a>

## 10. The Binary Tree Decision Map

```
QUESTION TYPE                        TRAVERSAL    KEY TECHNIQUE        LC
──────────────────────────────────────────────────────────────────────────
Max depth / height                   Post-order   max(L,R)+1           104
Level-by-level output                BFS          queue + level_size   102
Mirror / invert                      Pre-order    swap L,R recurse     226
Check balance                        Post-order   -1 sentinel          110
Diameter (longest path)              Post-order   L_h+R_h global max   543
Path sum root-to-leaf                Pre-order    carry remaining sum  112
Right side view                      BFS          last node each level 199
Lowest common ancestor               Post-order   both subtrees found  236
```

<a id='11'></a>

## 11. Interview Cheat Sheet

**1. When to reach for each traversal:**

| Pattern | Traversal | Why |
|---------|-----------|-----|
| Height / depth / balance | Post-order | Need child results before parent |
| Level output / shortest path | BFS | Process floor by floor |
| Print/process top-down | Pre-order | Parent before children |
| BST sorted output | In-order | Left < root < right |

**2. Core templates — memorize these:**

```python
# POST-ORDER DFS (most common in interviews)
def dfs(node):
    if not node: return 0        # base case
    left  = dfs(node.left)
    right = dfs(node.right)
    return combine(node.val, left, right)

# BFS LEVEL ORDER
q = deque([root])
while q:
    for _ in range(len(q)):      # snapshot level size
        node = q.popleft()
        if node.left:  q.append(node.left)
        if node.right: q.append(node.right)

# GLOBAL MAX PATTERN (diameter, max path sum)
best = [0]
def dfs(node):
    if not node: return 0
    L, R = dfs(node.left), dfs(node.right)
    best[0] = max(best[0], L + R)   # update global
    return max(L, R) + 1            # return height
```

**3. Gotchas:**

```
❌  Forgetting base case: if not node: return 0
❌  Returning diameter (L+R) instead of height (max(L,R)+1) upward
❌  Using global variable instead of list for Python closure mutation
❌  BFS: forgetting to snapshot len(q) before inner loop
✅  -1 sentinel avoids O(n²) by combining height+balance in one pass
✅  Post-order = bottom-up = results flow from leaves to root
```

<a id='12'></a>

## 12. Summary Map

```
BINARY TREE
│
├── DFS (stack / recursion)
│     ├── Pre-order  (root→L→R)  — top-down, invert, serialize
│     ├── In-order   (L→root→R)  — BST sorted output
│     └── Post-order (L→R→root)  — height, balance, diameter, LCA
│
└── BFS (queue)
      └── Level-order — level output, right view, min depth

CHOOSE BY:
  need child results before parent  → post-order DFS
  process level by level            → BFS
  top-down carries state            → pre-order DFS
  BST property (sorted order)       → in-order DFS

THE RECURSION SKELETON:
  if not node: return base
  L = dfs(node.left)
  R = dfs(node.right)
  return combine(node.val, L, R)
```

---
*End of Binary Tree Master Guide — Sean Edition*